#  LogBR Transportes: Relatório de Evolução de Motoristas

### Contexto
O time de RH da LogBR quer montar um painel de acompanhamento de desempenho para decidir quais motoristas receberão bonificação trimestral.
A análise precisa mostrar, para cada motorista, como seu desempenho evoluiu ao longo do tempo, onde ele se posiciona relativamente aos colegas e qual foi seu melhor e pior momento histórico.

### Metadados

| Coluna esperada | Descrição |
| :--- | :--- |
| **motorista_nome** | Nome do motorista |
| **regiao** | Região |
| **entrega_id** | ID da entrega |
| **data_entrega** | Data da entrega |
| **dias_em_transito** | Dias reais |
| **pct_acumulado_prazo** | % acumulado de entregas no prazo do motorista até aquela entrega (ordenado por data) |
| **quartil_desempenho** | Quartil do motorista dentro da sua região, baseado no pct_acumulado_prazo da entrega mais recente |
| **primeira_entrega_dias** | dias_em_transito da primeira entrega do motorista |
| **melhor_entrega_dias** | Menor dias_em_transito já registrado pelo motorista até aquela entrega |
| **percent_rank_regiao** | Posição relativa do motorista na região (0 a 1), baseada no pct_acumulado_prazo mais recente |

---

### Regras de negócio:

* **Entrega no prazo** = `dias_em_transito <= prazo_dias`
* **pct_acumulado_prazo** deve crescer ou diminuir entrega a entrega, refletindo o histórico até aquele momento
* **quartil_desempenho** e **percent_rank_regiao** são calculados sobre o snapshot mais recente de cada motorista — use `QUALIFY` para isso
* O resultado final deve conter apenas a última entrega de cada motorista

### Critérios de sucesso

* **pct_acumulado_prazo** na primeira entrega deve ser 1.0 ou 0.0 (100% ou 0% com base em 1 entrega)
* **melhor_entrega_dias** na primeira linha deve ser igual a **dias_em_transito** dessa entrega
* **quartil_desempenho** deve ter valores entre 1 e 4
* **QUALIFY** deve ser usado para filtrar apenas a última entrega de cada motorista sem CTE extra ou subquery
* **percent_rank_regiao** do motorista com menor **pct_acumulado_prazo** na região deve ser 0.0

In [0]:
%sql
select

m.nome as motorista_nome,
m.regiao,
e.data_entrega,
e.dias_em_transito,
e.prazo_dias,
sum(if(e.dias_em_transito <= e.prazo_dias,1,-1)) over (partition by m.nome order by e.data_entrega rows between unbounded preceding and current row) as qt_entregas_no_prazo,
round(sum(if(e.dias_em_transito <= e.prazo_dias,1,-1)) over (partition by m.nome order by e.data_entrega rows between unbounded preceding and current row) / count(*)over (partition by e.motorista_id order by e.data_entrega rows between unbounded preceding and current row),2) as acumulado_entregas_prazo

from sandbox.projetos.motoristas as m
left join sandbox.projetos.entregas as e
on e.motorista_id = m.motorista_id
